# Proyecto Final – Detección de Fraude con Tarjetas de Crédito

Este notebook sigue el estilo de las prácticas: **explicación → código → resultados**. Se utiliza el dataset `creditcard.csv` para detectar transacciones fraudulentas.

## 1. Introducción

La detección de fraude es un problema crítico en el ámbito financiero. El fraude representa una parte muy pequeña de las transacciones, pero sus consecuencias son grandes. Por ello, necesitamos modelos capaces de identificar la clase minoritaria.

**Objetivos del proyecto:**

- Analizar el dataset y el desbalance de clases.
- Entrenar un modelo base (Regresión Logística) y un modelo más complejo (Random Forest).
- Evaluar con métricas adecuadas (precision, recall, F1 y ROC-AUC).

## 2. Descripción del dataset

El archivo `creditcard.csv` contiene transacciones reales:

- **Time**: segundos desde la primera transacción.
- **Amount**: importe de la transacción.
- **V1 – V28**: variables transformadas con PCA.
- **Class**: 0 (legítima), 1 (fraude).

In [ ]:
# Imports básicos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Carga del dataset
# Asegúrate de que creditcard.csv esté en el mismo directorio

df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
# Información rápida
print(df.shape)
df.info()

In [ ]:
# Desbalance de clases
print(df["Class"].value_counts())

In [ ]:
# Visualización del desbalance
plt.figure(figsize=(5, 4))
sns.countplot(x="Class", data=df)
plt.title("Distribución de clases")
plt.xlabel("Clase (0 = legítima, 1 = fraude)")
plt.ylabel("Nº transacciones")
plt.tight_layout()
plt.show()

**Nota:** en un dataset tan desbalanceado, la **accuracy** puede ser engañosa. Por eso usaremos métricas centradas en la clase minoritaria.

## 3. Preprocesado de datos

- Separamos variables (X) y etiqueta (y).
- Escalamos las columnas `Time` y `Amount`.
- Dividimos en train/test con estratificación.

In [ ]:
# Separación de variables y etiqueta
X = df.drop(columns=["Class"])
y = df["Class"]

# División estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Escalado simple para Time y Amount
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

## 4. Modelos utilizados

### 4.1 Regresión Logística (modelo base)

Modelo sencillo y rápido. Usamos `class_weight='balanced'` para compensar el desbalance.

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)

### 4.2 Random Forest (modelo más complejo)

Modelo basado en árboles, capaz de capturar relaciones no lineales. También se usa `class_weight='balanced'`.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf.fit(X_train, y_train)

## 5. Evaluación

Métricas utilizadas:

- **Precision**
- **Recall**
- **F1-score**
- **ROC-AUC**
- **Matriz de confusión**

In [ ]:
def evaluar_modelo(modelo, X_test, y_test, nombre="Modelo"):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"=== {nombre} ===")
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Matriz de confusión - {nombre}")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.figure(figsize=(4, 3))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title(f"Curva ROC - {nombre}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return precision, recall, f1, roc_auc

In [ ]:
metrics_log = evaluar_modelo(log_reg, X_test, y_test, "Regresión Logística")

In [ ]:
metrics_rf = evaluar_modelo(rf, X_test, y_test, "Random Forest")

## 6. Comparación de resultados

In [ ]:
results = pd.DataFrame(
    [metrics_log, metrics_rf],
    columns=["Precision", "Recall", "F1-score", "ROC-AUC"],
    index=["Regresión Logística", "Random Forest"]
)
results

## 7. Conclusiones

- El modelo que mejor funcione dependerá del equilibrio entre **precision** y **recall**.
- En fraude suele priorizarse el **recall**, para detectar el máximo número de fraudes.

**Limitaciones:**

- No se han ajustado hiperparámetros.
- Solo se ha usado un split train/test.

**Mejoras futuras:**

- Búsqueda de hiperparámetros.
- Uso de SMOTE u otras técnicas de balanceo.
- Probar modelos más avanzados.